In [8]:
import asyncio
async def say_hello(n:int) -> str:
    print(f"  👋 开始问候 {n}...")
    await asyncio.sleep(1)
    print(f"  ✅ {n} 问候完成！")
    return f"Hello, {n}!"
async def main():
    print("=== 异步程序启动 ===\n")
    hello = await say_hello('alice')
    print(f"\n返回值:{hello}")
    print("\n === 异步程序结束 ===")
if __name__ == "__main__":
    await main()


=== 异步程序启动 ===

  👋 开始问候 alice...
  ✅ alice 问候完成！

返回值:Hello, alice!

 === 异步程序结束 ===


In [ ]:
import asyncio
import time
async def fetch_data(task_id:int,delay:float) -> str:
    print(f"{task_id}任务开始,等待{delay}秒")
    await asyncio.sleep(delay)
    print(f'{task_id}任务完成)')
    return f"{task_id}"
async def main():
    delays = [2,1,3,1,2]


    print("=== 串行执行 ===")
    start = time.time()
    for i,d in enumerate(delays):
        await fetch_data(i,d)
    serial_time = time.time() - start
    print(f"串行执行耗时: {serial_time:.2f}秒\n")


    print("=== 并发执行 ===")
    start = time.time()
    results = await asyncio.gather(
        *[fetch_data(i,d) for i,d in enumerate(delays)]
    )
    parallel_time = time.time() - start
    print(f"结果:{results}")
    print(f"加速比:{serial_time/parallel_time:.1f}")
if __name__ == "__main__":
    await main()

=== 串行执行 ===
0任务开始,等待2秒
0任务完成)
1任务开始,等待1秒
1任务完成)
2任务开始,等待3秒
2任务完成)
3任务开始,等待1秒
3任务完成)
4任务开始,等待2秒
4任务完成)
串行执行耗时: 9.05秒

=== 并发执行 ===
0任务开始,等待2秒
1任务开始,等待1秒
2任务开始,等待3秒
3任务开始,等待1秒
4任务开始,等待2秒
1任务完成)
3任务完成)
0任务完成)
4任务完成)
2任务完成)
结果:['0', '1', '2', '3', '4']
加速比:3.0


In [6]:
import asyncio
async def background_job(name:str,duration:float):
    print(f"{name}任务启动")
    await asyncio.sleep(duration)
    print(f"{name}任务完成")
    return f"{name}的结果"
async def main():
    print("=== 异步任务 ===")
    task1 = asyncio.create_task(background_job("下载文件",2))
    task2 = asyncio.create_task(background_job("解析数据",1))
    task3 = asyncio.create_task(background_job("生成报告",3))

    print("主协程：任务已提交，我先做点别的事...")
    await asyncio.sleep(0.5)
    print("主协程：别的事做完了，等待所有任务...\n")

    results = await asyncio.gather(task1,task2,task3)
    print(f"所有结果{results}")
if __name__ == "__main__":
    await main()

=== 异步任务 ===
主协程：任务已提交，我先做点别的事...
下载文件任务启动
解析数据任务启动
生成报告任务启动
主协程：别的事做完了，等待所有任务...

解析数据任务完成
下载文件任务完成
生成报告任务完成
所有结果['下载文件的结果', '解析数据的结果', '生成报告的结果']


In [10]:
import asyncio
class AsyncDatabaseConnection:
    def __init__(self,db_name:str):
        self.db_name = db_name
        self.connected = False
    async def __aenter__(self):
        print(f"连接到数据库 {self.db_name}...")
        await asyncio.sleep(1)
        self.connected = True
        print(f"已连接到数据库 {self.db_name}")
        return self
    async def __aexit__(self,exc_type,exc_val,exc_tb):
        print(f"正在关闭数据库[{self.db_name}]")
        await asyncio.sleep(0.2)
        self.connected = False
        print(f"数据库[{self.db_name}]")
        return False
    async def query(self,sql:str) -> list:
        if not self.connected:
           raise RuntimeError('未连接!')
        print(f'执行查询{sql}')
        await asyncio.sleep(0.3)
        return [{'id':1,'name':'alice'},{'id':2,'name':'bob'}]
async def main():
    print("=== 异步上下文管理器演示 ===\n")
    async with AsyncDatabaseConnection("my_ai_db") as db:
        results = await db.query("SELECT * FROM users")
        print(f'查询结果:{results}')
    print("=== 资源已安全释放 ===") 
if __name__ == "__main__":
    await main()   

=== 异步上下文管理器演示 ===

连接到数据库 my_ai_db...
已连接到数据库 my_ai_db
执行查询SELECT * FROM users
查询结果:[{'id': 1, 'name': 'alice'}, {'id': 2, 'name': 'bob'}]
正在关闭数据库[my_ai_db]
数据库[my_ai_db]
=== 资源已安全释放 ===


In [4]:
import asyncio
import time
async def limited_fetch(sem:asyncio.Semaphore,task_id:int) -> str:
    async with sem:
        print(f"  ▶️  任务 {task_id} 开始执行")
        await asyncio.sleep(1)
        print(f"  ⏹️  任务 {task_id} 完成")
        return f"result_{task_id}"
async def main():
    print("=== Semaphore 限流演示（最多 3 个并发）===\n")
    sem = asyncio.Semaphore(3)
    total_tasks = 10
    start_time = time.time()
    tasks = [limited_fetch(sem,i) for i in range(total_tasks)]
    results = await asyncio.gather(*tasks)
    elapsed = time.time() - start_time
    print(f"\n📊 {total_tasks} 个任务，并发限制 3,总耗时: {elapsed:.2f} 秒")
    print(f"   理论耗时 ≈ ceil(10/3) × 1s = 4 秒")
    print(f"📦 结果数量: {len(results)}")
if __name__ == "__main__":
    await main()


=== Semaphore 限流演示（最多 3 个并发）===

  ▶️  任务 0 开始执行
  ▶️  任务 1 开始执行
  ▶️  任务 2 开始执行
  ⏹️  任务 0 完成
  ⏹️  任务 1 完成
  ⏹️  任务 2 完成
  ▶️  任务 3 开始执行
  ▶️  任务 4 开始执行
  ▶️  任务 5 开始执行
  ⏹️  任务 3 完成
  ⏹️  任务 4 完成
  ⏹️  任务 5 完成
  ▶️  任务 6 开始执行
  ▶️  任务 7 开始执行
  ▶️  任务 8 开始执行
  ⏹️  任务 6 完成
  ⏹️  任务 7 完成
  ⏹️  任务 8 完成
  ▶️  任务 9 开始执行
  ⏹️  任务 9 完成

📊 10 个任务，并发限制 3,总耗时: 4.04 秒
   理论耗时 ≈ ceil(10/3) × 1s = 4 秒
📦 结果数量: 10
